In [2]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/022026/Data/MEDS_MDPS/data/tuning/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,14,NaT,GENDER//Kvinde,NaN
1,14,2003-01-31 00:00:00,DOB,NaN
2,14,2016-06-21 13:00:00,P/ZZ0150,NaN
3,14,2016-06-23 00:00:00,D/DF459,NaN
4,14,2016-07-06 15:39:00,P/ZZ0184,NaN
5,14,2016-10-05 15:09:00,P/AAF22,NaN
6,14,2020-01-29 00:00:00,D/DF999,NaN
7,14,2020-01-29 09:49:00,P/BVAA00,NaN
8,14,2020-01-30 13:26:00,P/BVAA33A,NaN
9,14,2020-01-30 14:00:00,P/ZZ0184,NaN


In [3]:
len(df)

56364836

In [5]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 221803
The patients has M-medication Codes: 150771
The patients has D-diagnosis Codes: 221755
The patients has P-Procedure Codes: 212751
The patients has S-SKS Codes: 52216


In [8]:
subject_counts = df['subject_id'].value_counts()


In [9]:
subject_counts

684625     56440
132804     49787
1968194    49683
856501     49610
1394370    46260
           ...  
1821767        3
2165382        3
943460         3
1150587        3
1831164        2
Name: subject_id, Length: 221803, dtype: int64

In [10]:
p_Num = df[df['code'].str.startswith('S/', na=False)]

In [11]:
p_Num

,subject_id,time,code,numeric_value
119,26,2019-02-14 23:59:00,S/KLEF40 KLCB28 KLEF00,NaN
582,242,2020-01-28 23:59:00,S/KDJD20 KDMB20,NaN
660,354,2018-08-06 23:59:00,S/KCJE20,NaN
700,468,2019-01-03 23:59:00,S/KJEA01,NaN
815,508,2021-10-24 23:59:00,S/KQBA10,NaN
...,...,...,...,...
56350978,2217148,2017-12-18 23:59:00,S/KEBA10,NaN
56351188,2217148,2020-11-22 23:59:00,S/KEEC15,NaN
56354386,2217463,2021-05-10 23:59:00,S/KLAF11,NaN
56364094,2218002,2016-11-28 23:59:00,S/KQBE10,NaN


In [13]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('S/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('S/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only porcedure code: ", only_p_ids_to_exclude)


Number of patients with only porcedure code:  []


In [14]:
df_filtered = df[~df['code'].str.startswith('S/', na=False)]

In [15]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [16]:
subject_counts_MDS

684625     56440
132804     49787
1968194    49683
856501     49610
1394370    46260
           ...  
195271         3
1562924        3
1623533        3
440377         3
1831164        2
Name: subject_id, Length: 221803, dtype: int64

In [17]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [19]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [20]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [21]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [22]:
comparison_df

,subject_id,original_count,new_count,difference
1685,1719866,3405,3376,29
193,1966150,8991,8966,25
10413,712426,1131,1109,22
2240,992199,2949,2929,20
728,1429160,4971,4953,18
...,...,...,...,...
101688,369423,82,82,0
101689,524897,82,82,0
101690,1326469,82,82,0
101691,505025,82,82,0


In [23]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 169587


In [24]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [25]:
print(most_changed.head(10))


       subject_id  original_count  new_count  difference  abs_diff
1685      1719866            3405       3376          29        29
193       1966150            8991       8966          25        25
10413      712426            1131       1109          22        22
2240       992199            2949       2929          20        20
728       1429160            4971       4953          18        18
4799      1520096            1918       1900          18        18
9474      1394776            1215       1198          17        17
4266      1166108            2061       2048          13        13
6205         4568            1627       1614          13        13
19870     1438085             632        619          13        13


In [26]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [27]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDPS codes',
        'new_count': 'MDP codes'
    }
)


In [28]:
lowest_new_count_patients

,subject_id,MDPS codes,MDP codes,difference,abs_diff
215763,1660625,6,5,1,1
215833,1864799,6,5,1,1


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [29]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

221803

In [30]:
len(df_filtered)

56288629

In [31]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('S/', na=False)].copy()
print("kept rows MDP:", len(df_filtered), " / total:", len(df))


kept rows MDP: 56288629  / total: 56364836


In [32]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
#N_SHARDS = 45
#df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

#print("rows to write:", len(df_filtered))


In [33]:
import numpy as np
import os

N_SHARDS = 5   #45 for Whole # 36 when we have split
OUT_DIR = "./_tuningMDPS_withoutS_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 5 parquet files into ./_tuningMDPS_withoutS_sharded


In [ ]:
import pyarrow.parquet as pq

OUT_DIR = "./_tuningMDPS_withoutS_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


In [ ]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_tuningMDPS_withoutS_sharded"
DST_PREFIX = "Zahra/022026/Data/MEDS_MDP/data/tuning"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


In [ ]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])
